In [0]:
from pyspark.sql.functions import (
    explode, col, to_timestamp, current_timestamp, first, 
    when, lit, max as spark_max
)
from pyspark.sql.types import DoubleType
spark.conf.set("spark.sql.ansi.enabled", "false")


bronze_table = "weather_stream.bronze.weather_data"
silver_table = "weather_stream.silver.weather_data"


# 2. Определяем максимальное время события, уже обработанное в Silver

try:
    last_event_time = spark.table(silver_table).select(spark_max("event_time")).collect()[0][0]
    if last_event_time is None:
        last_event_time = "1900-01-01 00:00:00"
    print(f"Последнее обработанное время (из Silver): {last_event_time}")
except Exception as e:
    print(f"Silver таблица не найдена или пуста: {e}")
    last_event_time = "1900-01-01 00:00:00"


# 3. Читаем Bronze и разворачиваем массив values

df_raw = (spark.table(bronze_table)
          .select(explode("values").alias("val"))
          .select(
              col("val.timestampUtc").alias("timestamp_utc_raw"),
              col("val.device_id").alias("device_id"),
              col("val.metric").alias("metric"),
              col("val.value").alias("value")
          ))


# 4. Преобразование timestamp_utc_raw (миллисекунды) в event_time (timestamp)

seconds_double = col("timestamp_utc_raw").cast("double") / 1000.0
seconds_long = seconds_double.cast("long")

# Допустимый диапазон секунд для Spark timestamp (0001-01-01 ... 9999-12-31)
MIN_SECONDS = -62135596800   # 0001-01-01
MAX_SECONDS = 253402300799   # 9999-12-31

df_with_time = df_raw.withColumn(
    "event_time",
    when(
        (seconds_long >= MIN_SECONDS) & (seconds_long <= MAX_SECONDS),
        to_timestamp(seconds_long)
    ).otherwise(lit(None))
)

# Отбрасываем строки, где event_time не удалось преобразовать (опционально)
df_valid = df_with_time.filter(col("event_time").isNotNull())


# 5. Инкрементальный фильтр – оставляем только новые записи
df_new = df_valid.filter(col("event_time") > last_event_time)

row_count = df_new.count()
if row_count == 0:
    print("Новых данных нет. Завершение работы.")
    dbutils.notebook.exit("No new data")
else:
    print(f"Найдено {row_count} новых строк после разворачивания массива")


# 6. Pivot: превращаем строки метрик в колонки

pivot_df = (df_new.groupBy("event_time", "device_id")
            .pivot("metric")
            .agg(first("value")))


# 7. Приводим колонки-метрики к типу DOUBLE (если возможно)
metric_columns = [c for c in pivot_df.columns if c not in ("event_time", "device_id")]
for mc in metric_columns:
    pivot_df = pivot_df.withColumn(mc, col(mc).cast(DoubleType()))


# 8. Добавляем колонку inserted_at (время загрузки в Silver)

pivot_df = pivot_df.withColumn("inserted_at", current_timestamp())

# 9. Удаляем возможные дубликаты (по event_time и device_id)

pivot_df = pivot_df.dropDuplicates(["event_time", "device_id"])

# 10. Запись в Silver таблицу (режим append)

if not spark.catalog.tableExists(silver_table):
    print("Создаю пустую таблицу Silver перед записью...")
    pivot_df.limit(0).write.mode("overwrite").saveAsTable(silver_table)

pivot_df.write.mode("append").saveAsTable(silver_table)

print(f"✅ Silver таблица обновлена. Добавлено строк: {pivot_df.count()}")